# Calculate shared colour limits for maps

Define data limits so that maps can be drawn on the same colour scales for all datasets.

In [14]:
import os
import pandas as pd

In [15]:
dir_output = 'output'
# Scenario results:
file_results_comparison = 'results_full_comparison_taunworc_base'
file_geo = 'results_geo_outcomes'
file_results = 'results_full'
# Output:
file_all_vlim = 'vlims_full.csv'
file_vlim_summary = 'vlims_summary.csv'

In [16]:
scenarios = ['base', 'taunworc']

## Set up data to gather



In [17]:
cols_vlims = [
    'nearest_ivt_time_no_bias',
    'drip_ship_mt_time',
    'drip_ship_mt_travel_time',
    'drip_ship_mt_travel_time_with_delay',
    'drip_ship_mrs_0-2_improvement_over_no_treatment',
    'lvo_drip_ship_ivt_mt_mrs_0-2_improvement_over_no_treatment',
    'change_in_drip_ship_mt_travel_time',
    'change_in_drip_ship_mt_time',
    'change_in_drip_ship_mt_travel_time_with_delay',
    'change_in_lvo_drip_ship_ivt_mt_mrs_0-2',
]

In [18]:
df_vlims = pd.DataFrame(index=cols_vlims, dtype=float)

## Gather file names

In [19]:
list_of_files = []

for s in '', '_bias10min':
    s_bias = 'nobias' if len(s) == 0 else s
    
    f1 = file_results_comparison + s + '.csv'
    list_of_files.append(f1)

    for f in [file_geo, file_results]:
        for scen in ['_base', '_taunworc']:
            f2 = f + scen + s + '.csv'
            list_of_files.append(f2)

In [20]:
list_of_files

['results_full_comparison_taunworc_base.csv',
 'results_geo_outcomes_base.csv',
 'results_geo_outcomes_taunworc.csv',
 'results_full_base.csv',
 'results_full_taunworc.csv',
 'results_full_comparison_taunworc_base_bias10min.csv',
 'results_geo_outcomes_base_bias10min.csv',
 'results_geo_outcomes_taunworc_bias10min.csv',
 'results_full_base_bias10min.csv',
 'results_full_taunworc_bias10min.csv']

## Find vlims

Load in each dataset and gather data limits:

In [21]:
dfs_to_concat = []

for f in list_of_files:
    df = pd.read_csv(os.path.join(dir_output, f))
    # Gather relevant columns:
    cols = [c for c in df.columns if
            any([c.startswith(col_vlim) for col_vlim in cols_vlims])]
    # Record which target column each matches:
    cols_match = []
    for c in cols:
        cols_start = [v for v in cols_vlims if c.startswith(v)]
        # Move longest column to the front:
        cols_start = sorted(cols_start, key=len)[::-1]
        if len(cols_start) > 0:
            cols_match.append(cols_start[0])
        else:
            print('Problem:', c)
    if len(cols) != len(cols_match):
        print('Check matches')
        print(cols)
        print(cols_match)
    # Find minimum and maximum values:
    df = df[cols].describe().loc[['min', 'max']]
    df.loc['file'] = f
    df = df.transpose()
    df['col_vlim'] = cols_match
    df.index.name = 'col'
    df = df.reset_index()
    dfs_to_concat.append(df)

/tmp/ipykernel_14585/4212434559.py:4: DtypeWarning: Columns (5: nearest_ivt_unit_multiple_same_time_and_distance_options, 13: nearest_mt_unit_multiple_same_time_and_distance_options, 18: transfer_unit_multiple_same_time_and_distance_options) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(os.path.join(dir_output, f))
/tmp/ipykernel_14585/4212434559.py:4: DtypeWarning: Columns (5: nearest_ivt_unit_multiple_same_time_and_distance_options, 13: nearest_mt_unit_multiple_same_time_and_distance_options, 18: transfer_unit_multiple_same_time_and_distance_options) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(os.path.join(dir_output, f))
/tmp/ipykernel_14585/4212434559.py:4: DtypeWarning: Columns (5: nearest_ivt_unit_multiple_same_time_and_distance_options, 13: nearest_mt_unit_multiple_same_time_and_distance_options, 18: transfer_unit_multiple_same_time_and_distance_options) have mixed types. Specify dtyp

In [22]:
df_limits = pd.concat(dfs_to_concat, axis='rows', ignore_index=True)
df_limits = df_limits.sort_values('col_vlim')

In [23]:
display(df_limits.head())

,col,min,max,file,col_vlim
6,change_in_drip_ship_mt_time,-117.0,0.0,results_full_comparison_taunworc_base.csv,change_in_drip_ship_mt_time
33,change_in_drip_ship_mt_time,-136.0,0.0,results_full_comparison_taunworc_base_bias10mi...,change_in_drip_ship_mt_time
7,change_in_drip_ship_mt_travel_time,-57.0,0.0,results_full_comparison_taunworc_base.csv,change_in_drip_ship_mt_travel_time
34,change_in_drip_ship_mt_travel_time,-76.0,0.0,results_full_comparison_taunworc_base_bias10mi...,change_in_drip_ship_mt_travel_time
35,change_in_drip_ship_mt_travel_time_with_delay,-136.0,0.0,results_full_comparison_taunworc_base_bias10mi...,change_in_drip_ship_mt_travel_time_with_delay


Gather min and max values more simply:

In [24]:
df_vlim = pd.DataFrame(columns=['min', 'max'])
df_vlim.index.name = 'column'

for col_vlim, df in df_limits.groupby('col_vlim'):
    vmin = df['min'].min()
    vmax = df['max'].max()
    df_vlim.loc[col_vlim] = [vmin, vmax]

In [25]:
display(df_vlim)

,min,max
column,,
change_in_drip_ship_mt_time,-136.000,0.000
change_in_drip_ship_mt_travel_time,-76.000,0.000
change_in_drip_ship_mt_travel_time_with_delay,-136.000,0.000
change_in_lvo_drip_ship_ivt_mt_mrs_0-2,0.000,0.086
drip_ship_mrs_0-2_improvement_over_no_treatment,0.082,0.139
drip_ship_mt_time,170.000,362.000
drip_ship_mt_travel_time,0.000,132.000
drip_ship_mt_travel_time_with_delay,0.000,192.000
lvo_drip_ship_ivt_mt_mrs_0-2_improvement_over_no_treatment,0.050,0.172


## Save results

In [26]:
p = os.path.join(dir_output, file_all_vlim)
df_limits.to_csv(p, index=False)

p = os.path.join(dir_output, file_vlim_summary)
df_vlim.to_csv(p, index=True)